# Text Normalization — edit-distance repair

Token-level repair of residual OCR errors using Levenshtein distance and `pyspellchecker`.

**Why after T5:** the T5 stage fixes structural damage; this stage fixes what remains at the
character level.

**Second motivation — adversarial evasion:** hate speech routinely uses deliberate misspellings and
character substitution to defeat keyword filters. Edit-distance normalization partially recovers the
intended token.

**Known limitation:** the adversarial-robustness motivation was never tested against actual
adversarial inputs — it remains design intent, not demonstrated capability.


In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import os
import re

In [ ]:


# Adjust these paths based on your Drive structure
CSV_PATH = "/content/Annotated_data-Final_data.csv"
IMG_FOLDER_PATH = "/content/drive/MyDrive/LGBTQ Memes/unique_images"
SAVE_PATH = "/content/drive/MyDrive/LGBTQ Memesprocessed_data.csv"

# Load CSV
df = pd.read_csv(CSV_PATH)




In [ ]:
df.head()

In [ ]:
df.columns = df.columns.str.strip()



In [ ]:
df = df[(df['Annotator_1'] != -1) & (df['Annotator_2'] != -1)]
df = df[df['Annotator_1'] == df['Annotator_2']]


In [ ]:
# Drop rows with missing OCR text
df = df.dropna(subset=['Extracted Text'])

# Add final 'label' column
df['label'] = df['Annotator_1'].astype(int)

# Show results
print(df.head())
print(f"[INFO] Cleaned dataset size: {len(df)} rows")

In [ ]:
!pip install textblob


In [ ]:
from textblob import TextBlob

# Function to correct spelling mistakes
def correct_spelling(text):
    if pd.isnull(text):
        return ""
    return str(TextBlob(text).correct())

# Apply to the 'Extracted Text' column
df['corrected_text'] = df['Extracted Text'].apply(correct_spelling)

# Check the result
print(df[['Extracted Text', 'corrected_text']].head())

# we also have autocorrect and pyspellchecker


In [ ]:
!pip install pyspellchecker


In [ ]:
from spellchecker import SpellChecker

# Initialize the spell checker
spell = SpellChecker()

# Function to correct spelling mistakes and handle None or empty strings
def correct_spelling_pyspellchecker(text):
    if not text:
        return ""  # Return an empty string if the text is None or empty
    words = text.split()
    corrected_words = [spell.correction(word) for word in words]
    # Ensure no None values are returned in the list before joining
    corrected_words = [word if word is not None else "" for word in corrected_words]
    return " ".join(corrected_words)

# Drop rows where 'Extracted Text' is NaN
df = df.dropna(subset=['Extracted Text'])

# Apply to the 'Extracted Text' column
df['corrected_text'] = df['Extracted Text'].apply(correct_spelling_pyspellchecker)

# Check the result
print(df[['Extracted Text', 'corrected_text']].head())


In [ ]:
!pip install symspellpy


In [ ]:
!wget https://raw.githubusercontent.com/mammothb/symspellpy/master/symspellpy/frequency_dictionary_en_82_765.txt


In [ ]:
sym_spell.load_dictionary("/content/frequency_dictionary_en_82_765.txt", term_index=0, count_index=1)


In [ ]:
from symspellpy import SymSpell, Verbosity
import pandas as pd
import re
from tqdm import tqdm

# Initialize SymSpell
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

cache = {}
def correct_spelling_symspell(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    text = re.sub(r'[^\w\s]', '', text)
    corrected_words = []
    for word in text.split():
        if word in cache:
            corrected = cache[word]
        else:
            suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
            corrected = suggestions[0].term if suggestions else word
            cache[word] = corrected
        corrected_words.append(corrected)
    return " ".join(corrected_words)

# Example: Drop NaNs and apply with progress bar
df = df.dropna(subset=['Extracted Text'])
tqdm.pandas()
df['corrected_text'] = df['Extracted Text'].progress_apply(correct_spelling_symspell)

# Preview result
print(df[['Extracted Text', 'corrected_text']].head())


In [ ]:
df.shape

In [ ]:
!pip install Levenshtein

In [ ]:
import numpy as np
import Levenshtein

# Function to calculate Levenshtein Distance
def calculate_levenshtein(text1, text2):
    return Levenshtein.distance(text1, text2)

from tqdm import tqdm
tqdm.pandas()

mask = df['Extracted Text'] != df['corrected_text']
df.loc[mask, 'levenshtein_distance'] = df[mask].progress_apply(
    lambda row: Levenshtein.distance(row['Extracted Text'], row['corrected_text']), axis=1)
df['levenshtein_distance'] = df['levenshtein_distance'].fillna(0).astype(int)

# Print the results
print(df[['Extracted Text', 'corrected_text', 'levenshtein_distance']].head())


In [ ]:
average_levenshtein = df['levenshtein_distance'].mean()
print(f"Average Levenshtein Distance: {average_levenshtein:.2f}")


In [ ]:
# Calculate normalized Levenshtein distance
df['normalized_levenshtein'] = df.apply(
    lambda row: row['levenshtein_distance'] / max(len(row['Extracted Text']), len(row['corrected_text']))
    if max(len(row['Extracted Text']), len(row['corrected_text'])) > 0 else 0,
    axis=1
)

# Compute average normalized distance
average_normalized = df['normalized_levenshtein'].mean()
print(f"Average Normalized Levenshtein Distance: {average_normalized:.4f}")


In [ ]:
df.shape

In [ ]:
# Full image path
df['image_path'] = df['Filename'].apply(lambda x: os.path.join(IMG_FOLDER_PATH, x))


In [ ]:
# Clean OCR text
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text.strip()

df['clean_text'] = df['Extracted Text'].apply(clean_text)


In [ ]:
# Drop rows with missing or unresolved fields
df.dropna(subset=['label', 'clean_text'], inplace=True)


In [ ]:
# Optional: Check if image files actually exist
df = df[df['image_path'].apply(os.path.exists)]


In [ ]:
# Convert label to integer
df['label'] = df['label'].astype(int)


In [ ]:
# Save processed CSV
df.to_csv('/content/Memesprocessed_data.csv', index=False)
print(f"[INFO] Preprocessed data saved at: {'/content/Memesprocessed_data.csv'}")
